In [ ]:
import os
import cv2
import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt 
import matplotlib.patches as mpatches

from mapping import stenosis_colors_rgb

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

In [ ]:
path2metaframe = "./data/meta_frame.csv"
frame = pd.read_csv(path2metaframe)
print(frame.shape)

In [ ]:
# Количество боксов как в описании датасета
frame.groupby('cls').size()

### Случайная визуализация нескольких фреймов

In [ ]:
sub_gr = frame.groupby("frame").size()
sub_gr[sub_gr > 5]

In [ ]:
num_frames = 50

frames_names = frame['frame'].unique().tolist()
select_frames = np.random.choice(frames_names, size=num_frames)

(x, y) - левый верхний угол
w, h - ширина и высота, соответственно

In [ ]:
fig, axs = plt.subplots(num_frames, 1, figsize=(10, 6 * num_frames))

for idx, i_frame in enumerate(select_frames):
    # --- select sub frame ---
    sub_frame = frame[frame['frame'] == i_frame].reset_index()

    # --- read img ---
    img = cv2.imread(sub_frame.loc[0, 'path2frame'])
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    # --- add bboxes ---
    legends = {}
    for i, irow in sub_frame.iterrows():
        icls = irow['cls'] 
        
        if icls == '-': # if nonlesion frame
            break

        x = int(irow['x'])
        y = int(irow['y']) 
        w = int(irow['w']) 
        h = int(irow['h']) 
        pt1 = (x, y)
        pt2 = (x + w, y + h)
        icolor = stenosis_colors_rgb[icls]

        if icls not in legends:
            legends[icls] = mpatches.Patch(color=tuple(item/255.0 for item in icolor), label=icls)

        cv2.rectangle(img_rgb, pt1, pt2, color=icolor, thickness=2)


    # --- add legend ---
    legend_handles = [legends[item] for item in legends]
    axs[idx].legend(handles=legend_handles, loc="upper right", framealpha=0.8)

    # --- print img ---
    axs[idx].imshow(img_rgb)
    axs[idx].set_title(f"{i_frame}")

plt.tight_layout()
plt.show()